Import required Python packages for this notebook.


In [1]:
import importlib, subprocess, sys

def ensure_installed(package):
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

ensure_installed("pysheds")

Import required Python packages for this notebook.


In [5]:
import os
import shutil
import numpy as np
from scipy import constants
import xarray as xr

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

from scripts import umeshFcts as ufcts

# Create a global mesh for goSPL

Create an unstructured grid for a given cell width. The method relies on the UXarray and jigsaw libraries.

**In case where the mesh already exists it will not be recreated.**

Spherical mesh resolution km

| cell_width | edge_min  | edge_max | edge_mean | nodeNb |
| ---------- | ----------  | ---------- | ---------- | ---------- |
| 5 | 1.1 | 4.5 | 2.8 | 23632811 |
| 8 |  1.8 | 7.2 | 4.6  | 9236387 | 
| 10 | 2.2 | 8.9 | 5.7 | 5912778 |
| 15 | 3.3 | 13.1 | 8.6 | 2629742 |
| 20 | 4.5 | 18 | 11.5 | 1480168 |
| 25 | 5.6 | 22.4 | 14.4 | 947701 |
| 30 | 6.8 | 26.4 | 17.2 | 658525 |
| 35 | 8 | 30.5 | 20.1 |  484009 |

In [3]:
widthCell = 40
input_path = "input_"+str(widthCell) 

# Build the mesh
ufcts.buildGlobalMeshSimple(widthCell, input_path)

Step 1. Generate mesh with JIGSAW
Running: jigsaw mesh.jig
 
#------------------------------------------------------------
#
#   ,o, ,o,       /                                 
#    `   `  e88~88e  d88~\   /~~~8e Y88b    e    / 
#   888 888 88   88 C888         88b Y88b  d8b  /   
#   888 888 "8b_d8"  Y88b   e88~-888  Y888/Y88b/  
#   888 888  /        888D C88   888   Y8/  Y8/     
#   88P 888 Cb      \_88P   "8b_-888    Y    Y    
# \_8"       Y8""8D                             
#
#------------------------------------------------------------
# JIGSAW: an unstructured mesh generation library.  
#------------------------------------------------------------
 
  JIGSAW VERSION 0.9.14

  Reading CFG. file...

  CFG. data summary...

  GEOM-FILE = mesh.msh 
  MESH-FILE = mesh-MESH.msh 
  HFUN-FILE = mesh-HFUN.msh 
  INIT-FILE =  
  TRIA-FILE =  
  BNDS-FILE =  

  GEOM-SEED = 8 
  GEOM-PHI1 = 6.00e+01 
  GEOM-PHI2 = 6.00e+01 
  GEOM-ETA1 = 4.50e+01 
  GEOM-ETA2 = 4.50e+01 
  GEOM-FEAT = F

Run shell commands for file or mesh management.


In [4]:
!mv mesh.jig mesh.log mesh.msh mesh-MESH.msh mesh-HFUN.msh mesh_triangles.nc cellWidthVsLatLon.nc $input_path

## Map variables on the UGRID 

We will now map global variables on this unstructured grid. In goSPL, typical variables would be:

- elevation (in m)
- vertical and horizontal tectonic forcing (displacement rates in m/yr)
- precipitation (in m/yr)
- dynamic topography (in m/yr)

Usually they will be provided in the form of `netcdf` or `geotiff` files. In both cases, the `xarray` or `rioxarray` libraries will allow you to open those files conveniently.

> Here we will use a netcdf grid containing all of these variables (except dynamic topography) for a give time interval.

In [6]:
# Loading the nc regular file
ncgrid = xr.open_dataset('data/251Ma.nc')
ncgrid

<xarray.Dataset> Size: 21MB
Dimensions:  (lat: 721, lon: 1441)
Coordinates:
  * lat      (lat) float64 6kB -90.0 -89.75 -89.5 -89.25 ... 89.5 89.75 90.0
  * lon      (lon) float64 12kB -180.0 -179.8 -179.5 ... 179.5 179.8 180.0
Data variables:
    h        (lat, lon) float64 8MB ...
    rain     (lat, lon) float32 4MB ...
    te       (lat, lon) float64 8MB ...

In case the file contains more variables than the ones you need for goSPL, you can select only the necessary ones:

In [7]:
# Loading the UGRID file
ufile = input_path+'/mesh_'+str(widthCell)+'km.nc'
mapds = xr.open_dataset(ufile) 

# Perform the interpolation (bilinear) 
var_path = 'vars_'+str(widthCell)
var_name = 'step_251'
if os.path.exists(var_path):
    shutil.rmtree(var_path)
ufcts.inter2UGRID(ncgrid,mapds,var_path,var_name,type='face')
data_ds = xr.open_dataset(var_path + '/' + var_name + '.nc')

In the `var_path` folder, you will find interpolated variables for the the UGRID (one file per variable) 

In [8]:
# Extract nodes and faces information
n_nodes = mapds.dims['nCells']
ucoords = np.zeros((n_nodes, 3))
ucoords[:, 0] = mapds['xCell'].values
ucoords[:, 1] = mapds['yCell'].values
ucoords[:, 2] = mapds['zCell'].values
ufaces = mapds['cellsOnVertex'].values - 1 
print(f"Number of nodes: {len(ucoords)} | Number of faces: {len(ufaces)}")

# Get information about your mesh:
dcEdge = mapds['dcEdge'].values  # in metres
edge_min = np.round(dcEdge.min() /1000.+0.,2)
edge_max = np.round(dcEdge.max() /1000.+0.,2)
edge_mean = np.round(dcEdge.mean() /1000.+0.,2)
print("edge range (km): min ",edge_min," | max ",edge_max," | mean ",edge_mean)

Number of nodes: 370830 | Number of faces: 741656
edge range (km): min  31.32  | max  52.95  | mean  39.88


Save voronoi mesh for visualisation purposes


In [9]:
# Save voronoi mesh for visualisation purposes
saveVoro = False

if saveVoro:
    from mpas_tools.viz.paraview_extractor import extract_vtk
    extract_vtk(
            filename_pattern=ufile,
            variable_list='areaCell',
            dimension_list=['maxEdges=','nVertLevels=', 'nParticles='], 
            mesh_filename=ufile,
            out_dir=input_path, 
            ignore_time=True,
            # lonlat=True,
            xtime='none'
        )
    print("You could now visualise in Paraview (wireframe) the produced voronoi mesh!")
    print("This is a vtp mesh called: ", input_path+'/staticFieldsOnCells.vtp')

> You might want to check that everything went according to plan and look at the mesh and variables that will be used in goSPL.

To do so, we will build a `vtk` file that could be visualised in Paraview...

In [10]:
checkMesh = False

if checkMesh:
    import meshio

    paleovtk = input_path+"/init.vtk"

    vlist = list(data_ds.keys())
    vdata = []
    for k in vlist:
        vdata.append(data_ds[k].values)

    list_data = dict.fromkeys(el for el in vlist)
    list_data.update((k, vdata[i]) for i, k in enumerate(list_data))

    # Define mesh
    vis_mesh = meshio.Mesh(ucoords, {"triangle": ufaces}, 
                           point_data = list_data,
                        )
    # Write it disk
    meshio.write(paleovtk, vis_mesh)
    print("Writing VTK input file as {}".format(paleovtk))

Set up file paths and names for mesh and output files.


In [15]:
meshname = var_path+"/mesh"
np.savez_compressed(meshname, v=ucoords, c=ufaces, 
                    z=data_ds.h.data
                    )

Now we save the forcing conditions (displacement rates, tectonic, precipitation...). Here you have the option to also add the next time step elevation, this will then be used in goSPL to force the model to match with the next paleo-elevation for specific regions (by defining the `zfit` parameter in the input file).

In [16]:
forcname = var_path+"/forcing251"

np.savez_compressed(forcname, 
                    te=data_ds.te.data, 
                    r=data_ds.rain.data,
                    )


This cell is currently empty.
